# Story — صدور بستهٔ ساخت روی Google Colab

**چه کار می‌کند:** متن داستان → **نمایشنامه + پرامپت + کد موتور** (بدون ویدیو داخل Colab).

**چه کار نمی‌کند:** رندر ویدیو · Remotion Pro · Director Board.

ویدیو را خودت با فایل‌های داخل `export/engines/` بساز.

---

### آماده‌سازی
1. پوشه **`Story`** در Drive: `MyDrive/Story/`
2. این نوت‌بوک را در Colab باز کن و Run all

## ۱) اتصال Drive و مسیرها

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil

drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
# اگر پوشه جای دیگری است، همین مسیر را عوض کن:
SRC = MYDRIVE / 'Story'
ROOT = Path('/content/Story')
OUT = MYDRIVE / 'StoryOut'
OUT.mkdir(parents=True, exist_ok=True)


def resolve_story_root(preferred: Path) -> Path:
    """Find Story project root on Drive (preferred path or auto-search)."""
    if preferred.is_dir() and (preferred / '__main__.py').is_file():
        return preferred
    for main in MYDRIVE.rglob('__main__.py'):
        root = main.parent
        if (root / 'requirements.txt').is_file() and (root / 'tools').is_dir():
            return root
    raise FileNotFoundError(
        f'پوشه Story پیدا نشد. در Drive باید Story/__main__.py باشد.\n'
        f'مسیر فعلی SRC = {preferred}'
    )


SRC = resolve_story_root(SRC)

os.environ['STORY_ROOT'] = str(ROOT)
os.environ['ANIMATION_OUT_ROOT'] = str(OUT)
os.environ['STORY_CHECKPOINT_DIR'] = str(ROOT / '.story' / 'checkpoints')
os.environ['STORY_USE_LLM'] = '0'
os.environ['ANIMATION_DETERMINISTIC_SLIDES'] = '1'
os.environ['QUALITY_GATE_STRICT'] = '0'

if ROOT.exists():
    shutil.rmtree(ROOT)
shutil.copytree(SRC, ROOT)  # اجرا روی دیسک محلی Colab (سریع‌تر از Drive)

print('SRC (Drive) =', SRC)
print('ROOT (Colab) =', ROOT, '| main =', (ROOT / '__main__.py').is_file())
print('OUT =', OUT)

## ۲) نصب وابستگی‌ها (Python)

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path(os.environ.get('STORY_ROOT', '/content/Story'))
sys.path.insert(0, str(ROOT))
%cd {ROOT}

!pip install -q -r requirements.txt

from tools.story_pipeline import readiness_probe
import json
p = readiness_probe()
print(json.dumps({k: p.get(k) for k in ('ok','ffmpeg','remotion','node')}, indent=2))
assert p.get('ok'), 'Story ready probe failed'
print('Export-only — no in-tool render')

## ۳) سناریو + صدور بسته

Brief را عوض کن. خروجی در `MyDrive/StoryOut/<job_id>/`

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path(os.environ['STORY_ROOT'])
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from scripts.colab_personal import setup_env, export_personal

setup_env(out_root=os.environ.get('ANIMATION_OUT_ROOT'))

BRIEF = """
قهرمان وارد سالن آرام می‌شود.

بعد با شوک واکنش نشان می‌دهد چون نامه می‌سوزد.

آرام بیرون می‌رود.
""".strip()

SECONDS = 24
CHARACTER = None

result = export_personal(BRIEF, seconds=SECONDS, character_path=CHARACTER, use_llm=False)
art = result.get('artifacts') or {}
export_root = Path(str(art.get('drive_copy') or art.get('export_root') or ''))
screenplay = export_root / 'screenplay.md'

print('export_root =', export_root)
print('screenplay exists =', screenplay.is_file())
assert export_root.is_dir() and screenplay.is_file(), 'صدور شکست خورد'

from IPython.display import Markdown, display
display(Markdown(screenplay.read_text(encoding='utf-8')[:6000]))
print('کپی Drive:', art.get('drive_copy'))

## ۴) نکات

- خروجی در `MyDrive/StoryOut/<job_id>/` (نمایشنامه، پرامپت، `engines/`)
- رندر ویدیو: راهنمای `engines/slideshow/guide.md` یا `engines/remotion/guide.md`
- پوشه Drive عوض شد → سلول ۱ را دوباره Run کن
- **تخته کارگردان (Streamlit):** بعد از brief، پیش‌نویس سناریو را تأیید/ویرایش کنید؛ سپس storyboard ساخته می‌شود (`STORY_CHECKPOINT_DIR` برای resume)